# VPO from scratch

<a href="https://colab.research.google.com/github/ryanboldi/vpo/blob/main/notebooks/01_vpo_from_scratch.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

This notebook explains VPO (Vector Policy Optimization) from the ground up, in plain NumPy. No GPU, no
training framework. You only need a rough idea of what VPO is for; everything else is built step by step.

Here is the situation. We have a base model that we want to improve with reinforcement learning. One
training step looks like this:

1. Give the model a prompt. The model produces a response.
2. A **verifier** checks the response and hands back scores.
3. The training method (GRPO, VPO, ...) turns those scores into an update that makes good responses more
   likely next time.

The detail that motivates VPO: for many tasks the verifier naturally returns **several numbers, not
one**. A maze route is scored on reaching the exit, collecting gold, collecting diamonds, and avoiding
lava. A program is scored on each test case separately. So the reward is a **vector**, one entry per
objective.

GRPO needs a single number, so it averages the vector and trains on the average. VPO keeps the vector.
VPO also changes what the model produces: the model is asked to write **m answers inside one response**,
and the response is judged as a *set*. This notebook builds both methods and shows what the vector and
the set buy you.

Plan:

1. Build a tiny fake verifier and a tiny fake model.
2. Explain and implement GRPO.
3. Explain and implement VPO.
4. Experiment 1: objectives that trade off. VPO learns to cover them; GRPO cannot.
5. Experiment 2: a single answer can satisfy every objective, but it is hard to find. VPO finds it;
   GRPO gets stuck.

## The fake verifier

A real verifier runs the model's text through checkers (does the route reach the exit? does the program
pass test 3?). To study the training method we do not need any of that machinery. Our toy task has
exactly **three possible answers**, numbered 0, 1, 2, and the verifier is a lookup table: give it an
answer, get back two scores, one per objective, each between 0 and 1.

The table is built so the two objectives pull in different directions:

- **answer 0** is perfect on objective 0 and useless on objective 1
- **answer 1** is the opposite
- **answer 2** scores 0.6 on both: a decent compromise, and the highest *average*

One thing to notice before we train anything. Picture a user who cares about the objectives with some
personal weighting. If they lean even moderately toward objective 0, their favorite is answer 0. If they
lean toward objective 1, answer 1. Only users very close to an exact 50/50 weighting prefer the
compromise, answer 2. Keep that picture in mind; VPO is built around it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# The fake verifier: one row per answer, one column per objective.
VERIFIER = np.array([
    [1.0, 0.0],   # answer 0: perfect on objective 0, useless on objective 1
    [0.0, 1.0],   # answer 1: the opposite
    [0.6, 0.6],   # answer 2: decent on both, and the highest average
])
N_ANSWERS, K_OBJ = VERIFIER.shape


def verify(answer):
    """Score one answer. Returns a vector with K_OBJ entries, each in [0, 1]."""
    return VERIFIER[answer]


fig, ax = plt.subplots(figsize=(4, 4))
ax.scatter(VERIFIER[:, 0], VERIFIER[:, 1], s=120, c=['C0', 'C1', 'C2'])
for i, (x, y) in enumerate(VERIFIER):
    ax.annotate(f"answer {i}", (x, y), textcoords="offset points", xytext=(8, 8))
ax.set_xlabel('objective 0 score'); ax.set_ylabel('objective 1 score')
ax.set_title('What the verifier returns for each answer')
ax.set_aspect('equal'); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

## The fake model

A real base model writes free-form text. For understanding the training method we can replace it with
something tiny, because GRPO and VPO only ever do three things with a model: sample a response from it,
score the response, and nudge the model's probabilities. So our model is just a table of probabilities
over the three answers, stored as logits (raw scores that we squash with a softmax to get
probabilities).

One more ingredient, and it matters. During VPO training the model is prompted to write **m answers
inside a single response**. (In the real codebase the prompt literally asks for m numbered solutions,
and the response contains `<route_1>` through `<route_m>` tags.) Our fake model therefore has m logit
tables, one per **answer slot**: `theta[j]` controls what goes in slot j. Producing one response means
sampling one answer from each slot.

Because every slot has its own table, the model is *able* to put different answers in different slots,
deliberately. Whether it actually learns to do that is entirely up to the training method. We use m = 4.

In [ ]:
M = 4   # answers per response


def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def sample_response(theta, rng):
    """Produce one response: one answer per slot. theta has shape (M, N_ANSWERS).

    Returns (answers, scores): answers has shape (M,), scores has shape (M, K_OBJ),
    one verifier vector per answer in the response.
    """
    p = softmax(theta)
    answers = np.array([rng.choice(N_ANSWERS, p=p[j]) for j in range(theta.shape[0])])
    return answers, np.array([verify(a) for a in answers])

## How GRPO works

GRPO (Group Relative Policy Optimization) trains the model like this, one prompt at a time:

1. **Sample a group.** Generate a group of responses to the same prompt (we use 16 per step).
2. **Give each response one number.** The verifier returned a vector per answer, and our responses
   contain m answers each, so GRPO squashes all of it: it takes the average of every score in the
   response. That single number is the response's reward.
3. **Compare within the group.** A reward of 0.6 means nothing on its own; what matters is whether it
   beats the other responses to the same prompt. So GRPO standardizes the rewards within the group:
   subtract the group's mean, divide by the group's standard deviation. The result is the **advantage**.
   Positive means better than the group average, negative means worse.
4. **Update the model.** Make every choice inside a positive-advantage response more likely, and every
   choice inside a negative-advantage response less likely, in proportion to the advantage. (This is the
   REINFORCE rule; we implement it in the training loop section below.)

The property to notice is in step 2: the only signal that survives is "average score". *Which* objective
an answer was good at is erased before the model ever sees a gradient.

In [ ]:
def grpo_advantage(responses):
    """GRPO: one number per response (the average of all its scores),
    standardized within the group."""
    rewards = np.array([scores.mean() for scores in responses])
    return (rewards - rewards.mean()) / (rewards.std() + 1e-6)

## How VPO works

VPO changes step 2 and nothing else. It keeps the score vectors and asks a different question about the
response.

Think about the users this model will eventually serve. Different users care about the objectives
differently. We can describe a user by a **preference vector** `w`: one non-negative weight per
objective, summing to 1. A user with preference `w` values an answer with score vector `r` at the
weighted sum `w · r`. A user with `w = (1, 0)` only cares about objective 0; a user with
`w = (0.5, 0.5)` cares about both equally.

VPO scores one response (a set of m answers) like this:

1. **Draw many random preferences.** `rng.dirichlet` with all parameters equal to 1 draws weight
   vectors uniformly at random from all valid weightings. Each draw is one imaginary user.
2. **Each user picks their favorite answer in the response**: the max over the m answers of `w · r`.
3. **Average that favorite's value over all sampled users.** This average is the response's reward.

In words: *if a random user showed up, how happy would they be with the best of these m answers, on
average?* A response scores high only if, for every kind of user, at least one of its answers serves
that user well. That is a statement about the set as a whole, and it is exactly why VPO rewards putting
*different* answers in different slots.

Steps 3 and 4 of GRPO (standardize within the group, REINFORCE update) are unchanged. The two methods
differ only in the single number they assign to a response.

In [ ]:
def vpo_advantage(responses, n_w=64, rng=None, alpha=1.0):
    """VPO: reward = average over random user preferences of the value of the
    response's best answer for that user. Standardized within the group."""
    rng = rng or np.random.default_rng(0)
    k = np.asarray(responses[0]).shape[1]        # number of objectives
    w = rng.dirichlet(np.full(k, alpha), size=n_w)   # n_w random users, shape (n_w, k)
    rewards = np.array([
        (scores @ w.T)        # value of every answer for every user, shape (m, n_w)
        .max(axis=0)          # each user picks their favorite answer, shape (n_w,)
        .mean()               # average happiness across users
        for scores in responses
    ])
    return (rewards - rewards.mean()) / (rewards.std() + 1e-6)

## The training loop, shared by both methods

The update rule is REINFORCE. Take a response with advantage `A`. For every slot j in that response,
where the model picked answer `a`, nudge the slot's logits so that picking `a` again becomes more likely
if `A > 0` and less likely if `A < 0`, with the size of the nudge proportional to `A`.

Note that all m slots of a response share the same `A`, because the response was scored as a whole.
Under VPO this shared signal is what lets the slots coordinate: if a response scored well *because* slot
0 held answer 0 and slot 1 held answer 1, the whole combination gets reinforced together.

In [ ]:
def reinforce_grad(theta, answers_list, advantages):
    """REINFORCE gradient, summed over every response in the group."""
    p = softmax(theta)
    grad = np.zeros_like(theta)
    for answers, A in zip(answers_list, advantages):
        for j, a in enumerate(answers):
            onehot = np.zeros(N_ANSWERS); onehot[a] = 1.0
            grad[j] += A * (onehot - p[j])
    return grad / len(answers_list)


def train(estimator_fn, n_steps=300, group_size=16, m=M, lr=0.1, seed=0):
    rng = np.random.default_rng(seed)
    theta = np.zeros((m, N_ANSWERS))
    history = []                       # per-slot probabilities over time
    for _ in range(n_steps):
        answers_list, responses = [], []
        for _ in range(group_size):
            answers, scores = sample_response(theta, rng)
            answers_list.append(answers); responses.append(scores)
        A = (estimator_fn(responses) if estimator_fn is grpo_advantage
             else estimator_fn(responses, rng=rng))
        theta += lr * reinforce_grad(theta, answers_list, A)
        history.append(softmax(theta).copy())
    return np.array(history)           # shape (n_steps, m, N_ANSWERS)

## Experiment 1: objectives that trade off

We train the same model twice, once with each method, and watch what each answer slot learns.

What should happen?

- **GRPO** scores responses by their overall average. Answer 2 has the best average (0.6, versus 0.5 for
  the corner answers), so the average is maximized by filling *every* slot with answer 2. All slots
  should collapse to the same answer.
- **VPO** scores responses by random-user happiness with the best answer in the set. A set that mixes
  answers 0 and 1 serves every user well: whoever shows up, one of the corners is close to what they
  want. The math agrees: a {0, 1} mix gives an average best-answer value of 0.75, while an all-answer-2
  set gives 0.6. So the slots should *specialize*, some taking answer 0 and some answer 1.

In [ ]:
hist_grpo = train(grpo_advantage, seed=0)
hist_vpo = train(vpo_advantage, seed=0)

fig, axes = plt.subplots(2, M, figsize=(3 * M, 6), sharex=True, sharey=True)
for row, (hist, name) in enumerate([(hist_grpo, 'GRPO'), (hist_vpo, 'VPO')]):
    for slot in range(M):
        ax = axes[row, slot]
        for i in range(N_ANSWERS):
            ax.plot(hist[:, slot, i], label=f'answer {i}')
        ax.set_ylim(0, 1); ax.grid(alpha=0.3)
        ax.set_title(f'{name}, answer slot {slot}')
        if slot == 0:
            ax.set_ylabel('P(answer)')
        if row == 1:
            ax.set_xlabel('training step')
axes[0, -1].legend(fontsize=8, loc='center right')
plt.tight_layout(); plt.show()

print("Most likely response after training (one answer per slot):")
print(f"  GRPO: {[int(hist_grpo[-1, j].argmax()) for j in range(M)]}   "
      f"(answer 2 four times: the compromise, repeated)")
print(f"  VPO : {[int(hist_vpo[-1, j].argmax()) for j in range(M)]}   "
      f"(a mix of answers 0 and 1: the set covers both objectives)")

**Reading the plot.** Each panel is one answer slot; each line is the probability of one answer. Under
GRPO, every slot converges to answer 2: the model's response is the same safe compromise written four
times. Under VPO, the slots split between answers 0 and 1: the model has learned to spend its m answers
covering the two objectives.

Both methods used the same model, the same sampling, and the same update rule. The only difference was
the single number assigned to each response before standardizing. Averaging produced collapse; the
random-user best-of-set produced division of labor.

## Look at one response

The plot above shows the policy's probabilities. Here is the object the policy actually produces: one
response, m answers, drawn in score space. The arrows are a few randomly drawn user preferences. For
each preference we circle the answer in the response that this user would pick as their favorite.

Under VPO, different users pick different answers from the same response. Under GRPO every user is stuck
with the compromise, because the response contains nothing else.

In [ ]:
rng = np.random.default_rng(1)
ws = rng.dirichlet([1, 1], size=4)        # 4 random users

fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharex=True, sharey=True)
for ax, hist, name in [(axes[0], hist_grpo, 'GRPO'), (axes[1], hist_vpo, 'VPO')]:
    response = np.array([hist[-1, j].argmax() for j in range(M)])   # most likely response
    scores = VERIFIER[response]                                     # shape (m, K_OBJ)
    jitter = rng.normal(0, 0.012, size=scores.shape)                # separate overlapping dots
    ax.scatter(VERIFIER[:, 0], VERIFIER[:, 1], s=260, facecolors='none',
               edgecolors='lightgray', linewidths=1.5, zorder=1)
    ax.scatter((scores + jitter)[:, 0], (scores + jitter)[:, 1], s=140, c='C3', zorder=3,
               label='the m answers in the response')
    for w in ws:
        ax.annotate('', xy=w * 1.05, xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color='C7', alpha=0.6))
        favorite = int((scores @ w).argmax())
        ax.scatter(*scores[favorite], s=420, facecolors='none', edgecolors='C2',
                   linewidths=2, zorder=2)
    for i, (x, y) in enumerate(VERIFIER):
        ax.annotate(f'answer {i}', (x, y), textcoords='offset points', xytext=(8, 8),
                    color='gray')
    ax.set_title(f'{name}: one response, plus 4 random users')
    ax.set_xlabel('objective 0 score'); ax.set_aspect('equal'); ax.grid(alpha=0.3)
axes[0].set_ylabel('objective 1 score'); axes[0].legend(loc='upper right')
plt.tight_layout(); plt.show()
print("Green circles mark each user's favorite answer within the response.")

## Why several answers per response matter

What happens to VPO if the model writes only one answer per response, m = 1?

The max in "each user picks their favorite answer" has nothing to choose between. Each user's value is
just `w · r` for the single answer, and averaged over uniformly random users that equals the plain
average of the score vector (every weight averages out to 1/K). Which is exactly GRPO's number.

So with one answer per response, **VPO assigns the same reward as GRPO** and the two methods become the
same algorithm. Writing several answers at once is not an implementation detail of VPO; it is the thing
that gives the max something to do. We can verify it directly:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
for ax, m in zip(axes, [1, 4]):
    hist = train(vpo_advantage, m=m, seed=0)
    marg = hist.mean(axis=1)               # average P(answer) across slots
    for i in range(N_ANSWERS):
        ax.plot(marg[:, i], label=f'answer {i}')
    ax.set_title(f'VPO with m={m}' + (' (same as GRPO: collapses to answer 2)' if m == 1
                                      else ' (covers answers 0 and 1)'))
    ax.set_xlabel('training step'); ax.set_ylim(0, 1); ax.grid(alpha=0.3); ax.legend()
axes[0].set_ylabel('average P(answer) over slots')
plt.tight_layout(); plt.show()

## Experiment 2: one answer can satisfy everything, but it is hard to find

In experiment 1 no single answer could win on both objectives, so VPO won by covering them with a set.
Coding is the opposite regime: one correct program passes *every* test, so a single answer can max all
objectives at once. If that answer is easy to find, both methods find it and tie. The interesting case
is when it is hard: hidden behind a **deceptive valley**, where partial progress looks *worse* on
average than not trying.

Here is a tiny model of that. An answer is now a **bitstring** with K blocks of s bits each. Think of K
parts of a program, where each part needs s lines to be right. The verifier scores objective k by
looking only at block k:

- block all zeros: score 0.8 (an easy, safe stub)
- each bit you turn on *lowers* the score (a half-finished part fails in ugly ways)
- all s bits on: score 1.0 (the part is fully correct)

The single perfect answer is all-ones: every objective scores 1.0. But every path from the safe stub to
the perfect answer passes through bitstrings that score *worse* than the stub. Let's draw the shape of
one block's score first:

In [ ]:
def trap(u, s):
    """Score of one block with u of its s bits turned on. 0.8 at u=0 (safe stub),
    decreasing through a valley, then 1.0 at u=s (fully correct)."""
    return 1.0 if u == s else 0.8 * (s - 1 - u) / (s - 1)


s_demo = 3
fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(range(s_demo + 1), [trap(u, s_demo) for u in range(s_demo + 1)], 'o-')
ax.set_xticks(range(s_demo + 1))
ax.set_xlabel('bits turned on in the block'); ax.set_ylabel('objective score')
ax.set_title('The deceptive valley (one block, s = 3)')
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

What should happen?

- **GRPO** follows the average of all objectives. From a random starting point, turning a bit *off*
  raises the average, so the model slides into the all-zeros stub on every block and sits at 0.8
  forever. The only way out goes downhill first, and the average never wants to go downhill.
- **VPO** has an escape route. A user whose preference concentrates on objective k is made happy by
  *any* answer in the set whose block k is correct, even if the rest of that answer is terrible. So it
  pays for one slot to specialize on one block at a time and cross that block's valley, while the other
  slots keep the rest of the score up. The set assembles full coverage block by block, and usually some
  slot ends up holding the perfect all-ones answer itself.

We use K = 4 objectives, s = 3 bits per block, the same m = 4 answer slots, and the **same two advantage
functions as before**. Only the model's action space changed: each slot now emits K x s independent
bits instead of picking one of three answers. We run 8 random seeds per method and track, after each
step, the random-user happiness with the model's most likely response (the same quantity VPO optimizes,
evaluated with fresh random users).

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def verify_bits(bits, K, s):
    """The verifier for experiment 2. bits has K*s entries; objective k is the
    trap score of block k. Returns a vector with K entries."""
    return np.array([trap(int(bits[k * s:(k + 1) * s].sum()), s) for k in range(K)])


def train_bits(estimator_fn, K=4, s=3, m=M, group_size=32, n_steps=500, lr=0.3, seed=0):
    """Same training loop as before; each answer is now K*s independent bits."""
    rng = np.random.default_rng(seed)
    L = K * s
    logit = np.zeros((m, L))                                   # m answer slots, L bits each
    curve = []
    for _ in range(n_steps):
        p = sigmoid(logit)
        bits = (rng.random((group_size, m, L)) < p).astype(float)
        responses = [np.array([verify_bits(bits[g, j], K, s) for j in range(m)])
                     for g in range(group_size)]               # each one is (m, K)
        A = (estimator_fn(responses) if estimator_fn is grpo_advantage
             else estimator_fn(responses, rng=rng))
        grad = np.zeros_like(logit)
        for g in range(group_size):
            grad += A[g] * (bits[g] - p)                       # REINFORCE for bits
        logit += lr * grad / group_size
        # evaluate: random-user happiness with the most likely response
        likely = (sigmoid(logit) > 0.5).astype(float)
        scores = np.array([verify_bits(likely[j], K, s) for j in range(m)])
        users = rng.dirichlet(np.ones(K), size=1000)
        curve.append(float((scores @ users.T).max(0).mean()))
    return np.array(curve), scores


SEEDS = range(8)
curves, last_set = {}, {}
for name, est in [('GRPO', grpo_advantage), ('VPO', vpo_advantage)]:
    runs = [train_bits(est, seed=sd) for sd in SEEDS]
    curves[name] = np.array([c for c, _ in runs])
    last_set[name] = runs[0][1]

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for name, color in [('GRPO', 'C3'), ('VPO', 'C0')]:
    C = curves[name]
    ax.plot(C.mean(0), color=color, lw=2, label=name)
    ax.fill_between(range(C.shape[1]), C.min(0), C.max(0), color=color, alpha=0.15)
ax.axhline(0.8, ls='--', c='gray', lw=1)
ax.text(8, 0.808, 'safe stub (all zeros)', color='gray', fontsize=8)
ax.axhline(1.0, ls=':', c='green', lw=1)
ax.text(8, 0.965, 'perfect answer (all ones)', color='green', fontsize=8)
ax.set_xlabel('training step'); ax.set_ylabel('random-user happiness with best answer')
ax.set_title('Experiment 2: a perfect answer hidden behind a deceptive valley')
ax.set_ylim(0.5, 1.06); ax.grid(alpha=0.3); ax.legend(loc='center right')
plt.tight_layout(); plt.show()

for name in ['GRPO', 'VPO']:
    fin = curves[name][:, -1]
    print(f"{name}: final value = {fin.mean():.3f} +/- {fin.std():.3f}   "
          f"|  reached the perfect answer in {(fin > 0.99).mean() * 100:.0f}% of seeds")

**What happened.** Both methods start at the safe stub, scoring 0.8 on everything. GRPO never leaves:
every exit lowers the average before it raises it, so the average-following gradient points back into
the stub, in 100% of seeds. VPO escapes in 100% of seeds. A user preference concentrated on objective k
rewards whichever slot has block k correct, regardless of that answer's other blocks, so specializing
pays where averaging punishes. The slots cross the valleys one block at a time, and the final set covers
everything, including a slot holding the perfect all-ones answer.

This is the subtler half of the VPO story. Even in tasks where a single answer can satisfy every
objective, so a diverse set looks unnecessary at first glance, the set is what makes the hard answer
*reachable* during training.

## Summary, and the real codebase

VPO makes two changes to GRPO, and we saw what each buys:

1. **Keep the reward as a vector** and score a response by the happiness of random imaginary users with
   the best answer in it. When objectives trade off, this makes the model cover them with a diverse set
   instead of collapsing to a bland compromise (experiment 1).
2. **Let the model write m answers per response** and judge the set as a whole. Without this, VPO
   reduces exactly to GRPO. And in hard tasks where one perfect answer exists behind a deceptive valley,
   the set is what lets training reach it at all (experiment 2).

Mapping the toy to the real code in this repo:

- The m answers per response come from a prompt rewrite that asks the model for m numbered solutions
  (`vpo/augment.py::AugmentedDataset`; the per-task rewrite specs live in `vpo_tasks/<task>.py`).
- The verifier is each task's reward function, which produces the per-answer score matrix.
- The advantage we wrote here is `vpo/utils/vpo.py::vpo_advantage`: the same average-over-users,
  max-over-the-set computation, in PyTorch, with an optional Sobol sampler that draws the random users
  more evenly for lower variance.

Next steps:

- [`02_advantage_explorer.ipynb`](02_advantage_explorer.ipynb) pokes at the real `vpo_advantage`
  function on synthetic score matrices.
- [`03_train_musique_lora.ipynb`](03_train_musique_lora.ipynb) trains a real LLM on the MuSiQue
  task with LoRA, end to end, and reproduces the paper's best@k comparison against GRPO.